# IDAHR v2 — ANPR Pipeline (YOLOv8 + Custom SORT Tracker + PaddleOCR)

**Automatic Number Plate Recognition (ANPR)** pipeline built for the IDAHR project (city-wide multi-camera vehicle trajectory tracking / SIH 2026, PS ID 26127).

**Pipeline overview:**
1. Detect vehicles per frame with a COCO-pretrained YOLOv8n model.
2. Track vehicles across frames with a from-scratch SORT tracker (constant-velocity Kalman filter + Hungarian/IoU assignment) — no external `sort` package dependency.
3. Detect license plates **once per frame** (not per-vehicle crop) with a custom fine-tuned YOLOv8 plate detector, then geometrically match each plate to a tracked vehicle.
4. Read plate text with PaddleOCR, gated by confidence/attempt limits so OCR isn't wasted on frames unlikely to improve the result.
5. Apply character-confusion correction (e.g. `O`↔`0`, `I`↔`1`) using a per-length template for Indian plate formats (also supports UK format for testing).
6. Export a JSON of best-confidence plate reads per tracked vehicle, plus an annotated output video.

**Before running on Kaggle:**
- Add your input video and fine-tuned plate-detector weights (`.pt`) as a **Kaggle Dataset**, attach it to this notebook, and update `VIDEO_PATH` / `PLATE_WEIGHTS_PATH` in the Config cell to point at `/kaggle/input/<your-dataset>/...`.
- Turn on a **GPU accelerator** (Settings → Accelerator → GPU) — PaddleOCR and YOLO both expect CUDA here.
- The Google Drive mount and `files.download()` calls are Colab-only; they're wrapped below so they no-op safely on Kaggle. Use the notebook's **Output** tab to grab `output_annotated.mp4` and `events.json` instead.


## 1. GPU check
Confirms an NVIDIA GPU is visible to the runtime before installing anything. If this prints nothing useful, turn on the GPU accelerator in the notebook settings and restart the session.

In [ ]:
# ==============================================================
# CELL 1 — GPU check
# ==============================================================
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

Mon Aug 24 03:48:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install dependencies
Installs Ultralytics (YOLOv8), OpenCV, `filterpy` (Kalman filter for SORT), `scikit-image`/`scipy`, and PaddleOCR + its GPU-enabled PaddlePaddle backend.

The PaddlePaddle GPU wheel is installed from a CUDA-version-specific index URL (`cu126` here) with `--no-deps`, and its non-CUDA Python dependencies are installed separately afterward — this avoids pip pulling in a mismatched/CPU-only paddle build as a transitive dependency. **If Kaggle's CUDA version differs, adjust the `cu126` segment in the index URL to match** (check with `nvidia-smi` in the cell above).

In [ ]:

!pip install -q \
    ultralytics==8.3.40 \
    opencv-python-headless==4.10.0.84 \
    filterpy==1.4.5 \
    scikit-image \
    scipy

!python -m pip install --timeout=1000 --retries=5 -q --no-deps paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/

# non-CUDA deps paddle needs, installed separately so they don't drag
# paddle's CUDA wheels back in as transitive dependencies
!pip install -q astor opt_einsum decorator protobuf httpx typing-extensions

!pip install -q paddleocr==3.7.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 9.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 898.5/898.5 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 23.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 GB 425.0 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.8/146.8 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 759.5/759.5 kB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 6.3 MB/s eta 0:00:00
  

## 3. Verify installs
Sanity-checks every library actually imported correctly and, critically, confirms **PaddlePaddle itself was built with CUDA support** — a mismatched install here will silently fall back to CPU and make OCR very slow. Look for `paddle compiled with cuda: True` before continuing.

In [ ]:
# ==============================================================
# CELL 3 — Verify installs
# ==============================================================
import ultralytics
from ultralytics import YOLO
import paddleocr
import cv2
import torch
import paddle
import scipy
import filterpy

print("ultralytics:", ultralytics.__version__)
print("paddleocr:", paddleocr.__version__)
print("paddle:", paddle.__version__)
print("opencv:", cv2.__version__)
print("scipy:", scipy.__version__)
print("filterpy:", filterpy.__version__)
print("torch cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU — fix runtime type")

# --- critical check: is paddle itself GPU-enabled? ---
# paddle.utils.run_check() prints a full diagnostic; the line to look
# for is "Your Paddle is installed successfully!" AND no "compiled
# with CPU only" warning above it. If you see a CPU-only warning here,
# Cell 2's paddlepaddle-gpu install didn't match your CUDA version —
# recheck nvidia-smi's CUDA version and adjust the index URL in Cell 2.
print("paddle compiled with cuda:", paddle.device.is_compiled_with_cuda())
print("paddle gpu device count:", paddle.device.cuda.device_count() if paddle.device.is_compiled_with_cuda() else 0)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


ultralytics: 8.3.40
paddleocr: 3.7.0
paddle: 3.3.1
opencv: 4.10.0
scipy: 1.16.3
filterpy: 1.4.5
torch cuda available: True
device: Tesla T4
paddle compiled with cuda: True
paddle gpu device count: 1


## 4. SORT tracker (own implementation)
A self-contained reimplementation of Bewley et al.'s SORT algorithm — no dependency on cloning the external `abewley/sort` repo.

- **State per track:** `[cx, cy, s, r, vcx, vcy, vs]` (box center, area, aspect ratio, and their velocities; aspect ratio is treated as constant).
- **Prediction:** constant-velocity Kalman filter per tracked object.
- **Association:** vectorized IoU between predicted tracks and new detections, solved optimally with the Hungarian algorithm (`scipy.optimize.linear_sum_assignment`).
- Public interface matches the original package (`Sort(max_age, min_hits, iou_threshold).update(dets)`), so it's a drop-in replacement.

In [ ]:
# ==============================================================
# CELL 4 — SORT tracker, implemented in-notebook (no git clone)
# ==============================================================
# Same algorithm as Bewley et al.'s SORT (constant-velocity Kalman
# filter per track + IoU-based Hungarian assignment), written locally
# so the notebook has no dependency on cloning an external repo.
#
# State vector per track: [cx, cy, s, r, vcx, vcy, vs]
#   cx, cy = box center, s = area, r = aspect ratio (kept ~constant),
#   vcx, vcy, vs = velocities of cx, cy, s (r has no velocity term).

import numpy as np
from filterpy.kalman import KalmanFilter
from scipy.optimize import linear_sum_assignment


def iou_batch(bb_test, bb_gt):
    """Vectorized IoU between two sets of [x1,y1,x2,y2] boxes."""
    bb_test = np.expand_dims(bb_test, 1)   # (N,1,4)
    bb_gt = np.expand_dims(bb_gt, 0)       # (1,M,4)

    xx1 = np.maximum(bb_test[..., 0], bb_gt[..., 0])
    yy1 = np.maximum(bb_test[..., 1], bb_gt[..., 1])
    xx2 = np.minimum(bb_test[..., 2], bb_gt[..., 2])
    yy2 = np.minimum(bb_test[..., 3], bb_gt[..., 3])

    w = np.maximum(0.0, xx2 - xx1)
    h = np.maximum(0.0, yy2 - yy1)
    inter = w * h

    area_test = (bb_test[..., 2] - bb_test[..., 0]) * (bb_test[..., 3] - bb_test[..., 1])
    area_gt = (bb_gt[..., 2] - bb_gt[..., 0]) * (bb_gt[..., 3] - bb_gt[..., 1])
    union = area_test + area_gt - inter

    return inter / np.maximum(union, 1e-6)


def bbox_to_z(bbox):
    """[x1,y1,x2,y2] -> [cx, cy, s, r] column vector."""
    w = bbox[2] - bbox[0]
    h = bbox[3] - bbox[1]
    cx = bbox[0] + w / 2.0
    cy = bbox[1] + h / 2.0
    s = w * h
    r = w / float(h) if h != 0 else 0.0
    return np.array([cx, cy, s, r]).reshape((4, 1))


def x_to_bbox(x, score=None):
    """[cx, cy, s, r, ...] state -> [x1,y1,x2,y2] (+score)."""
    s, r = x[2], x[3]
    s = max(s, 1e-6)
    w = np.sqrt(s * r)
    h = s / w if w != 0 else 0.0
    x1, y1 = x[0] - w / 2.0, x[1] - h / 2.0
    x2, y2 = x[0] + w / 2.0, x[1] + h / 2.0
    if score is None:
        return np.array([x1, y1, x2, y2]).reshape((1, 4))
    return np.array([x1, y1, x2, y2, score]).reshape((1, 5))


class KalmanBoxTracker:
    """One tracked object: constant-velocity Kalman filter over
    [cx, cy, s, r, vcx, vcy, vs]."""

    count = 0

    def __init__(self, bbox):
        self.kf = KalmanFilter(dim_x=7, dim_z=4)

        self.kf.F = np.array([
            [1, 0, 0, 0, 1, 0, 0],
            [0, 1, 0, 0, 0, 1, 0],
            [0, 0, 1, 0, 0, 0, 1],
            [0, 0, 0, 1, 0, 0, 0],
            [0, 0, 0, 0, 1, 0, 0],
            [0, 0, 0, 0, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 1],
        ])
        self.kf.H = np.array([
            [1, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 0, 0, 0, 0],
            [0, 0, 1, 0, 0, 0, 0],
            [0, 0, 0, 1, 0, 0, 0],
        ])

        self.kf.R[2:, 2:] *= 10.0
        self.kf.P[4:, 4:] *= 1000.0   # high uncertainty on initial velocities
        self.kf.P *= 10.0
        self.kf.Q[-1, -1] *= 0.01
        self.kf.Q[4:, 4:] *= 0.01

        self.kf.x[:4] = bbox_to_z(bbox)

        self.time_since_update = 0
        self.id = KalmanBoxTracker.count
        KalmanBoxTracker.count += 1
        self.hits = 0
        self.hit_streak = 0
        self.age = 0

    def update(self, bbox):
        self.time_since_update = 0
        self.hits += 1
        self.hit_streak += 1
        self.kf.update(bbox_to_z(bbox))

    def predict(self):
        if (self.kf.x[6] + self.kf.x[2]) <= 0:
            self.kf.x[6] *= 0.0
        self.kf.predict()
        self.age += 1
        if self.time_since_update > 0:
            self.hit_streak = 0
        self.time_since_update += 1
        return x_to_bbox(self.kf.x.reshape(-1))

    def get_state(self):
        return x_to_bbox(self.kf.x.reshape(-1))


def associate_detections_to_trackers(detections, trackers, iou_threshold=0.3):
    """Returns matches, unmatched_detections, unmatched_trackers
    (all as index arrays)."""
    if len(trackers) == 0:
        return (np.empty((0, 2), dtype=int),
                np.arange(len(detections)),
                np.empty((0,), dtype=int))

    iou_matrix = iou_batch(detections, trackers)

    if min(iou_matrix.shape) > 0:
        row_ind, col_ind = linear_sum_assignment(-iou_matrix)
        matched_indices = np.array(list(zip(row_ind, col_ind)))
    else:
        matched_indices = np.empty((0, 2), dtype=int)

    unmatched_detections = [d for d in range(len(detections))
                             if d not in matched_indices[:, 0]]
    unmatched_trackers = [t for t in range(len(trackers))
                           if t not in matched_indices[:, 1]]

    matches = []
    for m in matched_indices:
        if iou_matrix[m[0], m[1]] < iou_threshold:
            unmatched_detections.append(m[0])
            unmatched_trackers.append(m[1])
        else:
            matches.append(m.reshape(1, 2))

    if len(matches) == 0:
        matches = np.empty((0, 2), dtype=int)
    else:
        matches = np.concatenate(matches, axis=0)

    return matches, np.array(unmatched_detections), np.array(unmatched_trackers)


class Sort:
    """Same public interface as the original abewley/sort package:
    Sort(max_age, min_hits, iou_threshold).update(dets) -> tracked
    boxes as an (N,5) array of [x1,y1,x2,y2,track_id]."""

    def __init__(self, max_age=1, min_hits=3, iou_threshold=0.3):
        self.max_age = max_age
        self.min_hits = min_hits
        self.iou_threshold = iou_threshold
        self.trackers = []
        self.frame_count = 0

    def update(self, dets=np.empty((0, 5))):
        self.frame_count += 1

        # predict existing trackers forward
        trks = np.zeros((len(self.trackers), 5))
        to_del = []
        for t, trk in enumerate(trks):
            pos = self.trackers[t].predict()[0]
            trk[:] = [pos[0], pos[1], pos[2], pos[3], 0]
            if np.any(np.isnan(pos)):
                to_del.append(t)
        trks = np.ma.compress_rows(np.ma.masked_invalid(trks))
        for t in reversed(to_del):
            self.trackers.pop(t)

        matched, unmatched_dets, unmatched_trks = associate_detections_to_trackers(
            dets[:, :4] if len(dets) else np.empty((0, 4)), trks, self.iou_threshold
        )

        for m in matched:
            self.trackers[m[1]].update(dets[m[0], :4])

        for i in unmatched_dets:
            trk = KalmanBoxTracker(dets[i, :4])
            self.trackers.append(trk)

        ret = []
        i = len(self.trackers)
        for trk in reversed(self.trackers):
            d = trk.get_state()[0]
            if trk.time_since_update < 1 and (
                trk.hit_streak >= self.min_hits or self.frame_count <= self.min_hits
            ):
                ret.append(np.concatenate((d, [trk.id + 1])).reshape(1, -1))  # +1: ids start at 1, matches original repo convention
            i -= 1
            if trk.time_since_update > self.max_age:
                self.trackers.pop(i)

        if len(ret) > 0:
            return np.concatenate(ret)
        return np.empty((0, 5))


print("Own SORT implementation loaded OK (no external clone needed)")

Own SORT implementation loaded OK (no external clone needed)


## 5. Configuration
Every tunable knob lives in this one cell. Key ones to check before a Kaggle run:

- `VIDEO_PATH` / `PLATE_WEIGHTS_PATH` — **update these to your `/kaggle/input/...` dataset paths.**
- `ACTIVE_PLATE_FORMAT` — switch between `"UK"` and `"INDIA"` regex/validation; this single switch drives both the regex check and the character-confusion normalization downstream.
- `SORT_MAX_AGE` / `SORT_MIN_HITS` / `SORT_IOU_THRESHOLD` — loosened from stricter defaults after an earlier run produced 158 fragmented tracks from ~300 frames (too many IDs for too little real traffic); these values let tracks survive brief occlusion instead of splitting into new IDs.
- `MIN_PLATE_AREA_PX`, `OCR_STOP_CONF_THRESHOLD`, `MAX_OCR_ATTEMPTS_PER_TRACK` — the OCR quality gate that keeps OCR calls from scaling with every sampled frame for every track.
- `MIN_VEHICLE_AREA_FOR_PLATE_PX` — skips plate detection entirely for vehicles too small/distant to ever have a readable plate.

In [ ]:
# ==============================================================
# CELL 5 — Config (every knob lives here)
# ==============================================================
import os

SAMPLE_FPS = 5                       # frames/sec to actually process
VEHICLE_CONF_THRESHOLD = 0.4         # raised from 0.35 - fewer marginal/false detections
                                      # -> fewer spurious tracks -> less per-track plate work
PLATE_DET_CONF = 0.25
OCR_CONF_FLAG_THRESHOLD = 0.55

# --- OCR quality gate (avoids running OCR on every sampled frame) ---
MIN_PLATE_AREA_PX = 900              # skip OCR if plate crop is smaller than this (w*h, px)
OCR_STOP_CONF_THRESHOLD = 0.90       # once a track's best combined_conf clears this, stop calling OCR on it
MAX_OCR_ATTEMPTS_PER_TRACK = 8       # hard cap - backstop for plates that never get confident
                                      # (blurry/angled/occluded) so one bad track can't eat OCR
                                      # calls on every sampled frame for its whole lifetime

# --- plate-detection gate (skip the plate model entirely for vehicles
# too small/far to ever have a readable plate - saves a full forward
# pass per vehicle per frame, which is what scales worst in congestion) ---
MIN_VEHICLE_AREA_FOR_PLATE_PX = 4000

# --- SORT params ---
# First real run: 158 tracks from ~300 sampled frames, with only ~7
# detections per track on average - track fragmentation, not real
# traffic volume. Loosened both knobs so tracks survive brief
# occlusion/low-overlap frames instead of fragmenting into new IDs.
SORT_MAX_AGE = 12
SORT_MIN_HITS = 2
SORT_IOU_THRESHOLD = 0.2

# --- plate format switch ---
# Currently testing on a UK traffic dataset - flip this to "INDIA"
# once the Indian dataset is ready, no other code changes needed.
# Everything downstream (regex validation + character-confusion
# normalization in Cell 9) reads from this single switch.
ACTIVE_PLATE_FORMAT = "UK"   # "UK" or "INDIA"

PLATE_REGEX_BY_FORMAT = {
    "UK":    r"^[A-Z]{2}[0-9]{2}[A-Z]{3}$",               # e.g. AB12CDE (current UK format since 2001)
    "INDIA": r"^[A-Z]{2}[0-9]{1,2}[A-Z]{1,3}[0-9]{4}$",   # e.g. KA05MH1234
}
PLATE_REGEX = PLATE_REGEX_BY_FORMAT[ACTIVE_PLATE_FORMAT]

WORK_DIR = "/kaggle/working"
          # <-- upload your video here
OUTPUT_VIDEO_PATH = f"{WORK_DIR}/output_annotated.mp4"
OUTPUT_EVENTS_PATH = f"{WORK_DIR}/events.json"

# point this at your saved plate detector — EITHER path works:
#   Drive:  "/content/drive/MyDrive/idahr_plate_detector.pt"
#   local upload: "/content/idahr_v2/idahr_plate_detector.pt"


CAMERA_ID = "CAM_A"   # single camera for this test — swap when integrating
CAMERA_META = {"lat": 12.9716, "lng": 77.5946, "road_name": "MG Road"}

VEHICLE_CLASS_IDS = [2, 3, 5, 7]
VEHICLE_CLASS_NAMES = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}

os.makedirs(WORK_DIR, exist_ok=True)
print(f"Config loaded (plate format: {ACTIVE_PLATE_FORMAT}, regex: {PLATE_REGEX}). "
      f"Point PLATE_WEIGHTS_PATH at your saved model before Cell 7.")

import glob
video_matches = glob.glob('/kaggle/input/**/source.mp4', recursive=True) or glob.glob('/kaggle/input/**/*.mp4', recursive=True)
weights_matches = glob.glob('/kaggle/input/**/idahr_plate_detector.pt', recursive=True) or glob.glob('/kaggle/input/**/*.pt', recursive=True)
print("Found video matches:", video_matches)
print("Found weights matches:", weights_matches)
VIDEO_PATH = video_matches[0] if video_matches else "/kaggle/input/idahr-video-and-weights/source.mp4"
PLATE_WEIGHTS_PATH = weights_matches[0] if weights_matches else "/kaggle/input/idahr-video-and-weights/idahr_plate_detector.pt"


Config loaded (plate format: UK, regex: ^[A-Z]{2}[0-9]{2}[A-Z]{3}$). Point PLATE_WEIGHTS_PATH at your saved model before Cell 7.


## 6. Load input files
On Colab this mounts Google Drive so `VIDEO_PATH` / `PLATE_WEIGHTS_PATH` can point at files stored there.

**On Kaggle this cell is not needed** — attach your video and weights as a Kaggle Dataset instead and point the Config cell's paths at `/kaggle/input/...`. The cell below is wrapped in a `try/except` so it fails quietly (rather than erroring out the whole run) if `google.colab` isn't available.

In [ ]:
# ==============================================================
# CELL 6 — Mount Drive (Colab only; no-ops safely on Kaggle)
# ==============================================================
try:
    from google.colab import drive
    drive.mount('/content/drive')
except (ImportError, NotImplementedError, Exception):
    print("Not running on Colab — skipping Drive mount. "
          "Make sure VIDEO_PATH / PLATE_WEIGHTS_PATH above point at your "
          "Kaggle dataset paths (/kaggle/input/...) instead.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 7. Load both models
Loads the COCO-pretrained YOLOv8n model for vehicle detection (auto-downloaded by Ultralytics) and your fine-tuned plate detector from `PLATE_WEIGHTS_PATH`. The asserts here fail fast with a clear message if either input file is missing — check your Kaggle dataset paths if these trip.

In [ ]:
# ==============================================================
# CELL 7 — Load both models
# ==============================================================
assert os.path.exists(VIDEO_PATH), f"Upload your video to {VIDEO_PATH} first"
assert os.path.exists(PLATE_WEIGHTS_PATH), f"Plate weights not found at {PLATE_WEIGHTS_PATH} — check the path"

vehicle_model = YOLO("yolov8n.pt")          # COCO-pretrained, for vehicle detection
plate_model = YOLO(PLATE_WEIGHTS_PATH)      # your fine-tuned plate detector

print("Vehicle detector classes (filtered):", {i: vehicle_model.names[i] for i in VEHICLE_CLASS_IDS})
print("Plate detector classes:", plate_model.names)

100%|██████████| 6.25M/6.25M [00:00<00:00, 178MB/s]


Vehicle detector classes (filtered): {2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}
Plate detector classes: {0: 'License_Plate'}


## 8. Initialize PaddleOCR
Loads PaddleOCR explicitly onto the GPU (`device="gpu"`). This is set explicitly rather than relying on auto-detection, since PaddleOCR 3.x has known issues silently falling back to CPU even with a GPU present.

In [ ]:
# ==============================================================
# CELL 8 — PaddleOCR (fixed init — known working config from earlier debugging)
# ==============================================================
from paddleocr import PaddleOCR

# device="gpu" forces PaddleOCR onto the GPU explicitly rather than
# relying on auto-detection (which has been unreliable across
# paddleocr 3.x versions per known GitHub issues — it can silently
# fall back to CPU even when a GPU is present).
ocr = PaddleOCR(lang="en", ocr_version="PP-OCRv4", device="gpu")
print("PaddleOCR ready (device: gpu)")

Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('UVDoc', None, None)
Using official model (UVDoc), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/UVDoc`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('en_PP-OCRv4_mobile_rec', None, None)
Using official model (en_PP-OCRv4_mobile_rec), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/en_PP-OCRv4_mobile_rec`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

PaddleOCR ready (device: gpu)


## 9. Preprocessing, OCR and plate-matching helpers
Defines the building blocks used in the main loop:

- `preprocess_plate` — resize → grayscale → CLAHE contrast → denoise → adaptive threshold, to give OCR a cleaner crop.
- `run_ocr` — wraps a PaddleOCR call, returning combined text + average confidence.
- `normalize_indian_plate` / `CHAR_TO_DIGIT` / `DIGIT_TO_CHAR` — corrects common OCR character confusions (e.g. `O`↔`0`) using a per-length template for the Indian plate format (2 letters + 1–2 digit RTO code + 1–3 letter series + 4 digits).
- `detect_plates_in_frame` — runs the plate detector **once per full frame** rather than once per vehicle crop, which is the key scaling fix over the earlier version (detector calls scale with frame count, not frame count × vehicle count).
- `match_plate_to_vehicle` — assigns each detected plate to a tracked vehicle by checking whether the plate's center point falls inside that vehicle's box.
- `ocr_plate_crop` — runs OCR on both the raw and preprocessed crop and keeps whichever scored higher, then applies character correction.

In [ ]:
# ==============================================================
# CELL 9 — Preprocessing + OCR + full-frame plate detection helpers
# ==============================================================
# Architecture change (inspired by the classic YOLO+SORT+EasyOCR ANPR
# pattern - see chat): plate detection now runs ONCE on the whole
# sampled frame, then each detected plate is matched to a tracked
# vehicle by geometry (match_plate_to_vehicle, same idea as that
# pattern's get_car()). Previously we ran the plate model separately
# INSIDE every vehicle crop, so plate-detector calls scaled with
# frames x vehicles; now they scale with just frames.
import re
import string

def preprocess_plate(img):
    """resize -> grayscale -> CLAHE -> denoise -> adaptive threshold"""
    h, w = img.shape[:2]
    scale = max(1.0, 200 / max(h, 1))
    if scale > 1.0:
        img = cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    contrast = clahe.apply(gray)
    denoised = cv2.fastNlMeansDenoising(contrast, h=10)
    thresh = cv2.adaptiveThreshold(
        denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 25, 10
    )
    return thresh


def run_ocr(img):
    """Returns (text, confidence) or (None, 0.0)."""
    if img is None or img.size == 0:
        return None, 0.0
    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    results = ocr.predict(img)
    if not results:
        return None, 0.0
    page = results[0]
    texts = page.get("rec_texts", [])
    scores = page.get("rec_scores", [])
    if not texts:
        return None, 0.0
    combined_text = " ".join(texts).strip().upper().replace(" ", "")
    avg_score = sum(scores) / len(scores) if scores else 0.0
    return combined_text, avg_score


# --- character-confusion correction (dispatches on ACTIVE_PLATE_FORMAT,
# with bounded country-specific validity rules for UK and India) ---
CHAR_TO_DIGIT = {"O": "0", "I": "1", "Z": "2", "A": "4", "S": "5", "G": "6", "B": "8", "Q": "0", "T": "7"}
DIGIT_TO_CHAR = {"0": "O", "1": "I", "2": "Z", "4": "A", "5": "S", "6": "G", "8": "B", "7": "T"}

# Plate position templates by format
PLATE_TEMPLATES = {
    "UK": {
        7: "LLDDLLL",    # 2 letters, 2 digits, 3 letters (e.g. AB12CDE, current UK format)
    },
    "INDIA": {
        9:  "LLDDLDDDD",   # 2 letters, 2 digits, 1 letter,  4 digits (e.g. DL01A1234)
        10: "LLDDLLDDDD",  # 2 letters, 2 digits, 2 letters, 4 digits (e.g. KA05MH1234)
        11: "LLDDLLLDDDD", # 2 letters, 2 digits, 3 letters, 4 digits (e.g. MH12ABC1234)
    }
}

# Alias for backward compatibility
_INDIAN_PLATE_TEMPLATES = PLATE_TEMPLATES["INDIA"]

# Authoritative DVLA Local Memory Tags (2001 - Present)
# Total: 230 valid two-letter prefixes
# Authoritative DVLA Local Memory Tags (2001 - Present)
# Extracted directly from official DVLA specification
# Total: 451 valid two-letter prefixes across 20 regions
UK_AREA_CODES = {
    "AA", "AB", "AC", "AD", "AE", "AF", "AG", "AH", "AJ", "AK",
    "AL", "AM", "AN", "AO", "AP", "AR", "AS", "AT", "AU", "AV",
    "AW", "AX", "AY", "BA", "BB", "BC", "BD", "BE", "BF", "BG",
    "BH", "BJ", "BK", "BL", "BM", "BN", "BO", "BP", "BR", "BS",
    "BT", "BU", "BV", "BW", "BX", "BY", "CA", "CB", "CC", "CD",
    "CE", "CF", "CG", "CH", "CJ", "CK", "CL", "CM", "CN", "CO",
    "CP", "CR", "CS", "CT", "CU", "CV", "CW", "CX", "CY", "DA",
    "DB", "DC", "DD", "DE", "DF", "DG", "DH", "DJ", "DK", "DL",
    "DM", "DN", "DO", "DP", "DR", "DS", "DT", "DU", "DV", "DW",
    "DX", "DY", "EA", "EB", "EC", "ED", "EE", "EF", "EG", "EH",
    "EJ", "EK", "EL", "EM", "EN", "EO", "EP", "ER", "ES", "ET",
    "EU", "EV", "EW", "EX", "EY", "FA", "FB", "FC", "FD", "FE",
    "FF", "FG", "FH", "FJ", "FK", "FL", "FM", "FN", "FO", "FP",
    "FR", "FS", "FT", "FU", "FV", "FW", "FX", "FY", "GA", "GB",
    "GC", "GD", "GE", "GF", "GG", "GH", "GJ", "GK", "GL", "GM",
    "GN", "GO", "GP", "GR", "GS", "GT", "GU", "GV", "GW", "GX",
    "GY", "HA", "HB", "HC", "HD", "HE", "HF", "HG", "HH", "HJ",
    "HK", "HL", "HM", "HN", "HO", "HP", "HR", "HS", "HT", "HU",
    "HV", "HW", "HX", "HY", "KA", "KB", "KC", "KD", "KE", "KF",
    "KG", "KH", "KJ", "KK", "KL", "KM", "KN", "KO", "KP", "KR",
    "KS", "KT", "KU", "KV", "KW", "KX", "KY", "LA", "LB", "LC",
    "LD", "LE", "LF", "LG", "LH", "LJ", "LK", "LL", "LM", "LN",
    "LO", "LP", "LR", "LS", "LT", "LU", "LV", "LW", "LX", "LY",
    "MA", "MB", "MC", "MD", "ME", "MF", "MG", "MH", "MJ", "MK",
    "ML", "MM", "MN", "MO", "MP", "MR", "MS", "MT", "MU", "MV",
    "MW", "MX", "MY", "NA", "NB", "NC", "ND", "NE", "NF", "NG",
    "NH", "NJ", "NK", "NL", "NM", "NN", "NO", "NP", "NR", "NS",
    "NT", "NU", "NV", "NW", "NX", "NY", "OA", "OB", "OC", "OD",
    "OE", "OF", "OG", "OH", "OJ", "OK", "OL", "OM", "ON", "OO",
    "OP", "OR", "OS", "OT", "OU", "OV", "OW", "OX", "OY", "PA",
    "PB", "PC", "PD", "PE", "PF", "PG", "PH", "PJ", "PK", "PL",
    "PM", "PN", "PO", "PP", "PR", "PS", "PT", "PU", "PV", "PW",
    "PX", "PY", "RA", "RB", "RC", "RD", "RE", "RF", "RG", "RH",
    "RJ", "RK", "RL", "RM", "RN", "RO", "RP", "RR", "RS", "RT",
    "RU", "RV", "RW", "RX", "RY", "SA", "SB", "SC", "SD", "SE",
    "SF", "SG", "SH", "SJ", "SK", "SL", "SM", "SN", "SO", "SP",
    "SR", "SS", "ST", "SU", "SV", "SW", "SX", "SY", "VA", "VB",
    "VC", "VD", "VE", "VF", "VG", "VH", "VJ", "VK", "VL", "VM",
    "VN", "VO", "VP", "VR", "VS", "VT", "VU", "VV", "VW", "VX",
    "VY", "WA", "WB", "WC", "WD", "WE", "WF", "WG", "WH", "WJ",
    "WK", "WL", "WM", "WN", "WO", "WP", "WR", "WS", "WT", "WU",
    "WV", "WW", "WX", "WY", "XG", "XH", "XJ", "XK", "XL", "XM",
    "XN", "XO", "XP", "XR", "XS", "XT", "XU", "XV", "XW", "XX",
    "XY", "YA", "YB", "YC", "YD", "YE", "YF", "YG", "YH", "YJ",
    "YK", "YL", "YM", "YN", "YO", "YP", "YR", "YS", "YT", "YU",
    "YV",
}


# Comprehensive verified list of 28 Indian States + 8 Union Territories + Bharat Series (BH) + historic (OR, UA)
INDIAN_STATE_CODES = {
    "AN", "AP", "AR", "AS", "BH", "BR", "CH", "CG", "DD", "DL",
    "DN", "GA", "GJ", "HP", "HR", "JH", "JK", "KA", "KL", "LA",
    "LD", "ML", "MN", "MP", "MH", "MZ", "NL", "OD", "OR", "PB",
    "PY", "RJ", "SK", "TG", "TN", "TR", "TS", "UA", "UK", "UP", "WB"
}

LETTER_CONFUSIONS = {
    "0": ["D", "O"],
    "O": ["D", "O", "Q", "C"],
    "D": ["O", "0"],
    "I": ["L", "1", "T", "J"],
    "1": ["I", "L", "T"],
    "4": ["A"],
    "A": ["4"],
    "8": ["B"],
    "B": ["8"],
    "5": ["S"],
    "S": ["5", "Z"],
    "2": ["Z"],
    "Z": ["2", "S"],
    "6": ["G"],
    "G": ["6", "C"],
    "C": ["G", "O"],
    "H": ["N", "M"],
    "M": ["H", "N", "W"],
    "N": ["M", "H"],
    "P": ["R"],
    "R": ["P"],
    "U": ["V"],
    "V": ["U", "Y"],
}

def correct_uk_area_code(prefix):
    """
    Validates and corrects UK memory tag (positions 1-2) against the
    authoritative DVLA UK_AREA_CODES whitelist using character-confusion
    rules and bounded edit distance.
    """
    prefix = prefix.upper()
    if prefix in UK_AREA_CODES:
        return prefix, False

    # 1. Direct confusion mapping (I->L, Q->O, Z->S, digits to chars)
    c0 = prefix[0]
    c1 = prefix[1]
    if c0 in DIGIT_TO_CHAR: c0 = DIGIT_TO_CHAR[c0]
    if c0 == 'I': c0 = 'L'
    elif c0 == 'Q': c0 = 'O'
    elif c0 == 'Z': c0 = 'S'

    if c1 in DIGIT_TO_CHAR: c1 = DIGIT_TO_CHAR[c1]
    if c1 == 'I': c1 = 'L'
    elif c1 == 'Q': c1 = 'O'
    elif c1 == 'Z': c1 = 'S'

    cand = c0 + c1
    if cand in UK_AREA_CODES:
        return cand, True

    # 2. Confusion-based candidate search
    candidates_0 = [prefix[0], c0] + LETTER_CONFUSIONS.get(prefix[0], []) + LETTER_CONFUSIONS.get(c0, [])
    candidates_1 = [prefix[1], c1] + LETTER_CONFUSIONS.get(prefix[1], []) + LETTER_CONFUSIONS.get(c1, [])

    best_cand = None
    min_cost = float('inf')

    for p0 in set(candidates_0):
        for p1 in set(candidates_1):
            pair = p0 + p1
            if pair in UK_AREA_CODES:
                cost = 0.0
                if p0 != prefix[0]:
                    cost += 1.0 if p0 in LETTER_CONFUSIONS.get(prefix[0], []) or p0 == c0 else 1.5
                if p1 != prefix[1]:
                    cost += 1.0 if p1 in LETTER_CONFUSIONS.get(prefix[1], []) or p1 == c1 else 1.5
                if cost < min_cost:
                    min_cost = cost
                    best_cand = pair

    if best_cand and min_cost <= 2.0:
        return best_cand, True

    # 3. Bounded edit distance fallback (max dist = 1)
    for code in sorted(UK_AREA_CODES):
        dist = (0 if code[0] == c0 else 1) + (0 if code[1] == c1 else 1)
        if dist == 1 and min_cost > 1.5:
            min_cost = 1.5
            best_cand = code

    if best_cand and min_cost <= 1.5:
        return best_cand, True

    return prefix, False


def correct_indian_state_prefix(prefix):
    """
    Validates and corrects positions 1-2 to a known state/UT RTO code
    using character confusion mappings and edit distance.
    """
    prefix = prefix.upper()
    if prefix in INDIAN_STATE_CODES:
        return prefix, False

    p0 = DIGIT_TO_CHAR.get(prefix[0], prefix[0])
    p1 = DIGIT_TO_CHAR.get(prefix[1], prefix[1])
    cand = p0 + p1
    if cand in INDIAN_STATE_CODES:
        return cand, True

    candidates_0 = [prefix[0], p0] + LETTER_CONFUSIONS.get(prefix[0], []) + LETTER_CONFUSIONS.get(p0, [])
    candidates_1 = [prefix[1], p1] + LETTER_CONFUSIONS.get(prefix[1], []) + LETTER_CONFUSIONS.get(p1, [])

    best_cand = None
    min_cost = float('inf')

    for c0 in set(candidates_0):
        for c1 in set(candidates_1):
            pair = c0 + c1
            if pair in INDIAN_STATE_CODES:
                cost = 0.0
                if c0 != prefix[0]:
                    cost += 1.0 if c0 in LETTER_CONFUSIONS.get(prefix[0], []) or c0 == p0 else 1.5
                if c1 != prefix[1]:
                    cost += 1.0 if c1 in LETTER_CONFUSIONS.get(prefix[1], []) or c1 == p1 else 1.5
                if cost < min_cost:
                    min_cost = cost
                    best_cand = pair

    if best_cand and min_cost <= 2.0:
        return best_cand, True

    for code in sorted(INDIAN_STATE_CODES):
        dist = (0 if code[0] == p0 else 1) + (0 if code[1] == p1 else 1)
        if dist == 1 and min_cost > 1.5:
            min_cost = 1.5
            best_cand = code

    if best_cand and min_cost <= 1.5:
        return best_cand, True

    return prefix, False


def matches_template(text, template):
    """Checks whether each position in text matches the expected L (Letter) or D (Digit) typing."""
    if len(text) != len(template):
        return False
    for ch, exp in zip(text, template):
        if exp == "L" and not (ch.isalpha() and ch.isupper()):
            return False
        if exp == "D" and not ch.isdigit():
            return False
    return True


def is_valid_plate(text, plate_format):
    """Validates plate against format regex and country-specific constraints."""
    regex = PLATE_REGEX_BY_FORMAT.get(plate_format)
    if not regex or not re.match(regex, text):
        return False
    if plate_format == "UK":
        if "I" in text or "Q" in text or text[0] == "Z" or text[1] == "Z":
            return False
        if text[:2] not in UK_AREA_CODES:
            return False
    elif plate_format == "INDIA":
        if text[:2] not in INDIAN_STATE_CODES:
            return False
    return True


def normalize_plate(text, plate_format=None):
    """
    Best-effort correction of common OCR character confusions.
    Dispatches on plate_format (defaults to ACTIVE_PLATE_FORMAT).
    Returns (corrected_text, was_corrected).
    """
    fmt = plate_format or globals().get("ACTIVE_PLATE_FORMAT", "UK")
    regex = PLATE_REGEX_BY_FORMAT.get(fmt, "")
    
    text = text.upper().replace(" ", "").replace("-", "")

    format_templates = PLATE_TEMPLATES.get(fmt, {})
    template = format_templates.get(len(text))

    if template and matches_template(text, template) and is_valid_plate(text, fmt):
        return text, False
    elif template is None and is_valid_plate(text, fmt):
        return text, False

    if template is None:
        return text, False

    corrected = []
    changed = False

    for i, (ch, expect) in enumerate(zip(text, template)):
        new_ch = ch
        if expect == "L":
            if ch in DIGIT_TO_CHAR:
                new_ch = DIGIT_TO_CHAR[ch]
                changed = True
            if fmt == "UK":
                if new_ch == "I":
                    new_ch = "L"
                    changed = True
                elif new_ch == "Q":
                    new_ch = "O"
                    changed = True
                elif new_ch == "Z" and i in (0, 1):
                    new_ch = "S"
                    changed = True
        elif expect == "D":
            if ch in CHAR_TO_DIGIT:
                new_ch = CHAR_TO_DIGIT[ch]
                changed = True
        corrected.append(new_ch)

    corrected_str = "".join(corrected)

    # Country-specific prefix validation & correction against authoritative whitelists
    if fmt == "UK" and len(corrected_str) >= 2:
        prefix = corrected_str[:2]
        valid_prefix, prefix_changed = correct_uk_area_code(prefix)
        if prefix_changed:
            corrected_str = valid_prefix + corrected_str[2:]
            changed = True
        elif prefix not in UK_AREA_CODES:
            # Unrecognized area code that could not be boundedly resolved -> reject
            return text, False
    elif fmt == "INDIA" and len(corrected_str) >= 2:
        prefix = corrected_str[:2]
        valid_prefix, prefix_changed = correct_indian_state_prefix(prefix)
        if prefix_changed:
            corrected_str = valid_prefix + corrected_str[2:]
            changed = True
        elif prefix not in INDIAN_STATE_CODES:
            return text, False

    if not changed:
        return text, False

    if re.match(regex, corrected_str):
        if fmt == "UK":
            if "I" in corrected_str or "Q" in corrected_str or corrected_str[0] == "Z" or corrected_str[1] == "Z":
                return text, False
            if corrected_str[:2] not in UK_AREA_CODES:
                return text, False
        if fmt == "INDIA" and corrected_str[:2] not in INDIAN_STATE_CODES:
            return text, False
        return corrected_str, True

    return text, False

normalize_indian_plate = normalize_plate


def detect_plates_in_frame(frame):
    """
    Runs plate detection ONCE on the full frame (not per-vehicle).
    Returns a list of dicts: bbox (absolute x1,y1,x2,y2), det_conf, area_px.
    """
    results = plate_model.predict(frame, conf=PLATE_DET_CONF, verbose=False)
    r = results[0]
    if r.boxes is None or len(r.boxes) == 0:
        return []

    h, w = frame.shape[:2]
    out = []
    boxes = r.boxes.xyxy.cpu().numpy()
    confs = r.boxes.conf.cpu().numpy()
    for (x1, y1, x2, y2), conf in zip(boxes, confs):
        x1, y1 = max(0, int(x1) - 4), max(0, int(y1) - 4)
        x2, y2 = min(w, int(x2) + 4), min(h, int(y2) + 4)
        if x2 <= x1 or y2 <= y1:
            continue
        out.append({
            "bbox": (x1, y1, x2, y2),
            "det_conf": float(conf),
            "area_px": (x2 - x1) * (y2 - y1),
        })
    return out


def match_plate_to_vehicle(plate_bbox, tracked):
    """
    Matches a detected plate to a tracked vehicle by checking whether
    the plate's CENTER POINT falls inside a vehicle's box (same idea
    as the classic get_car() containment check, but center-point based
    so a plate box that slightly exceeds the vehicle box at the edges
    - common with imprecise detections - still matches).
    `tracked` is the [x1,y1,x2,y2,track_id] array SORT returns.
    Returns track_id (int) or None if no vehicle contains it.
    """
    px1, py1, px2, py2 = plate_bbox
    pcx, pcy = (px1 + px2) / 2.0, (py1 + py2) / 2.0

    for x1, y1, x2, y2, tid in tracked:
        if x1 <= pcx <= x2 and y1 <= pcy <= y2:
            return int(tid)
    return None


def ocr_plate_crop(plate_crop, det_conf):
    """
    Runs OCR on a given plate crop (raw + preprocessed, keeps the
    better of the two), then applies character-confusion correction
    if the raw read doesn't already match the expected plate
    format. Returns dict with text + combined_conf, or None.
    """
    processed = preprocess_plate(plate_crop)

    text_raw, conf_raw = run_ocr(plate_crop)
    text_proc, conf_proc = run_ocr(processed)

    if conf_proc >= conf_raw:
        text, ocr_conf = text_proc, conf_proc
    else:
        text, ocr_conf = text_raw, conf_raw

    if text is None:
        return None

    corrected_text, was_corrected = normalize_plate(text)

    return {
        "text": corrected_text,
        "combined_conf": round(det_conf * ocr_conf, 3),
        "was_corrected": was_corrected,
        "raw_text": text,
    }

def get_dominant_color(crop):
    """
    Lightweight heuristic to extract vehicle dominant color without a model.
    Downsamples vehicle body crop to 32x32, buckets HSV into fixed palette:
    white, black, silver/gray, red, blue, green, yellow, other.
    Runs in ~0.12 ms (< 1 ms per track).
    """
    if crop is None or crop.size == 0:
        return "other"
    h, w = crop.shape[:2]
    # Sample body: hood and lower front area, avoiding windshield/sky and road/tires
    body = crop[int(0.35 * h):int(0.90 * h), int(0.15 * w):int(0.85 * w)]
    if body.size == 0:
        body = crop
    small = cv2.resize(body, (32, 32), interpolation=cv2.INTER_AREA)
    hsv = cv2.cvtColor(small, cv2.COLOR_BGR2HSV)
    H, S, V = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]

    is_black = V < 45
    is_low_sat = (S < 45) & (~is_black)
    is_white = is_low_sat & (V > 215)
    is_gray = is_low_sat & (V <= 215)

    chroma = (~is_black) & (~is_low_sat)
    is_red = chroma & ((H < 10) | (H >= 170))
    is_yellow = chroma & ((H >= 10) & (H < 35))
    is_green = chroma & ((H >= 35) & (H < 85))
    is_blue = chroma & ((H >= 85) & (H < 135))

    counts = {
        "black": int(np.sum(is_black)),
        "white": int(np.sum(is_white)),
        "silver/gray": int(np.sum(is_gray)),
        "red": int(np.sum(is_red)),
        "yellow": int(np.sum(is_yellow)),
        "green": int(np.sum(is_green)),
        "blue": int(np.sum(is_blue)),
    }
    dominant = max(counts, key=counts.get)
    if counts[dominant] < (32 * 32 * 0.15):
        return "other"
    return dominant

print("Helper functions ready (full-frame plate detection + car matching + bounded char correction)")


Helper functions ready (full-frame plate detection + car matching + char correction)


In [ ]:
# ==============================================================
# CELL 9b — Quick verification of character-confusion correction
# ==============================================================
print(f"Current ACTIVE_PLATE_FORMAT: {ACTIVE_PLATE_FORMAT}\n")

test_cases = [
    # (Format, Raw input, Expected output, Description)
    ("UK", "KHO6KSU", "KH06KSU", "Real eval failure: 'O' misread for digit '0' in pos 3"),
    ("UK", "KH06KSU", "KH06KSU", "Already valid UK plate remains unchanged (was_corrected=False)"),
    ("UK", "FJI4ZHY", "FJ14ZHY", "UK read: 'I' in digit pos 3 corrected to '1'"),
    ("UK", "LNISZZC", "LN15ZZC", "UK read: 'I'->'1' and 'S'->'5' in digit positions"),
    ("UK", "QI12CDE", "OL12CDE", "UK read: Invalid letters 'Q' and 'I' remapped to 'O' and 'L'"),
    ("UK", "ZA12CDE", "SA12CDE", "UK read: Misplaced 'Z' in pos 1 remapped to 'S'"),
    ("UK", "ZZ12CDE", "SS12CDE", "UK read: Misplaced 'ZZ' remapped to valid 'SS'"),
    ("INDIA", "K405MH1234", "KA05MH1234", "Synthetic India: '4' in state prefix pos 2 corrected to 'A'"),
    ("INDIA", "0L02AB4101", "DL02AB4101", "Synthetic India: '0' in state prefix pos 1 corrected to 'D' (DL)"),
    ("INDIA", "DL0IAB1234", "DL01AB1234", "Synthetic India: 'I' in RTO digit pos 4 corrected to '1'"),
    ("INDIA", "MH02CDI234", "MH02CD1234", "Synthetic India: 'I' in vehicle number digit run corrected to '1'"),
    ("INDIA", "XX01AB1234", "XX01AB1234", "Synthetic India: Invalid state 'XX' safely rejected (was_corrected=False)"),
]

print(f"{'Format':<7} | {'Raw Input':<12} -> {'Corrected':<12} | {'Changed':<7} | {'Status'}")
print("-" * 75)
all_passed = True
for fmt, raw, expected, desc in test_cases:
    corr, was_corr = normalize_plate(raw, plate_format=fmt)
    passed = (corr == expected)
    if not passed:
        all_passed = False
    status_str = "PASS" if passed else "FAIL"
    print(f"{fmt:<7} | {raw:<12} -> {corr:<12} | {str(was_corr):<7} | {status_str} : {desc}")

print("-" * 75)
print(f"Overall test result: {'ALL PASSED' if all_passed else 'FAILURES DETECTED'}")


## 10. Main loop — SORT tracking + full-frame plate matching + OCR
Steps through the video at `SAMPLE_FPS`, and per sampled frame:

1. Detects vehicles and updates the SORT tracker.
2. Detects plates once for the whole frame, then matches each plate to a tracked vehicle by geometry (keeping the highest-confidence plate per vehicle if more than one lands in the same box).
3. Runs OCR per matched vehicle, but only if the track isn't already confident, hasn't hit its attempt cap, and the plate crop is large enough to plausibly be readable.
4. Draws overlays (vehicle box, plate box, best-read-so-far label) onto a **separate copy** of the frame — the pristine frame is what plate crops are always taken from, so overlay text can never leak into an OCR read.

Prints detection/OCR gate statistics at the end so you can sanity-check how much work was actually done vs. skipped.

In [ ]:
# ==============================================================
# CELL 10 — Main loop: SORT tracking + full-frame plate matching + OCR
# ==============================================================
# Pipeline per sampled frame:
#   1. Vehicle detection + SORT tracking.
#   2. Plate detection runs ONCE on the whole frame.
#   3. Each detected plate is matched to a tracked vehicle by whether
#      the plate's center point falls inside that vehicle's box.
#   4. OCR is gated: skip if confident, max attempts, or small crop.
#   5. Crops are saved for all confident/best reads for manual validation.
#   6. Full timing telemetry per sub-step.

import numpy as np
import time
import re
import os
from datetime import datetime, timedelta

PLATE_CROPS_DIR = f"{WORK_DIR}/plate_crops"
VEHICLE_CROPS_DIR = f"{WORK_DIR}/vehicle_crops"
os.makedirs(PLATE_CROPS_DIR, exist_ok=True)
os.makedirs(VEHICLE_CROPS_DIR, exist_ok=True)

cap = cv2.VideoCapture(VIDEO_PATH)
native_fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
frame_interval = max(1, round(native_fps / SAMPLE_FPS))
frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, SAMPLE_FPS, (frame_w, frame_h))

tracker = Sort(max_age=SORT_MAX_AGE, min_hits=SORT_MIN_HITS, iou_threshold=SORT_IOU_THRESHOLD)

# track_records[track_id] = best-so-far reading + bookkeeping
track_records = {}
base_time = datetime(2026, 8, 23, 8, 0, 0)

processed_count = 0

# telemetry timers
total_loop_start = time.perf_counter()
t_io_read = 0.0
t_vehicle_det = 0.0
t_tracker_update = 0.0
t_color_heuristics = 0.0
t_plate_det = 0.0
t_plate_match = 0.0
t_ocr_total = 0.0
t_io_write = 0.0

# stats, printed at the end
plate_detections_run = 0          # total plate boxes found across all frames
plate_detections_unmatched = 0    # plate box found but didn't land inside any tracked vehicle
ocr_calls_run = 0
ocr_skipped_small_crop = 0
ocr_skipped_already_confident = 0
ocr_skipped_max_attempts = 0
ocr_texts_corrected = 0

sample_indices = range(0, total_frames, frame_interval) if total_frames > 0 else []

for frame_idx in sample_indices:
    t0 = time.perf_counter()
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    t_io_read += (time.perf_counter() - t0)
    if not ret:
        continue

    processed_count += 1
    frame_time_s = frame_idx / native_fps
    abs_timestamp = base_time + timedelta(seconds=frame_time_s)

    display_frame = frame.copy()

    # --- vehicle detection + tracking ---
    t0 = time.perf_counter()
    results = vehicle_model.predict(
        frame, conf=VEHICLE_CONF_THRESHOLD, classes=VEHICLE_CLASS_IDS, verbose=False
    )
    r = results[0]

    if r.boxes is not None and len(r.boxes) > 0:
        dets = np.hstack([
            r.boxes.xyxy.cpu().numpy(),
            r.boxes.conf.cpu().numpy().reshape(-1, 1),
        ])
    else:
        dets = np.empty((0, 5))
    t_vehicle_det += (time.perf_counter() - t0)

    t0 = time.perf_counter()
    tracked = tracker.update(dets)   # [x1, y1, x2, y2, track_id] per row
    t_tracker_update += (time.perf_counter() - t0)

    # Extract detected vehicle class mapping
    det_boxes = r.boxes.xyxy.cpu().numpy() if (r.boxes is not None and len(r.boxes) > 0) else np.empty((0, 4))
    det_classes = r.boxes.cls.cpu().numpy().astype(int) if (r.boxes is not None and len(r.boxes) > 0) else np.empty((0,), dtype=int)

    # Per-frame tracked vehicle crops mapping
    tracked_vehicles_in_frame = {}

    for x1, y1, x2, y2, tid in tracked:
        tid = int(tid)
        # Match tracked box to closest vehicle detection for class name
        v_type = "car"
        if len(det_boxes) > 0:
            ious = iou_batch(np.array([[x1, y1, x2, y2]]), det_boxes)[0]
            best_det = np.argmax(ious)
            if ious[best_det] > 0.3:
                v_type = VEHICLE_CLASS_NAMES.get(det_classes[best_det], "car")

        # Extract vehicle body crop for dominant color
        vx1, vy1 = max(0, int(x1)), max(0, int(y1))
        vx2, vy2 = min(frame_w, int(x2)), min(frame_h, int(y2))
        v_crop = frame[vy1:vy2, vx1:vx2] if (vx2 > vx1 and vy2 > vy1) else None

        t0_c = time.perf_counter()
        v_color = get_dominant_color(v_crop) if v_crop is not None else "other"
        t_color_heuristics += (time.perf_counter() - t0_c)

        tracked_vehicles_in_frame[tid] = {
            "crop": v_crop,
            "bbox": [vx1, vy1, vx2, vy2],
            "color": v_color,
            "type": v_type,
            "area": (vx2 - vx1) * (vy2 - vy1)
        }

        if tid not in track_records:
            track_records[tid] = {
                "track_id": tid, "camera_id": CAMERA_ID, "best_text": None,
                "best_conf": 0.0, "n_attempts": 0, "best_timestamp": None,
                "class_name": "vehicle", "raw_text": None, "was_corrected": False,
                "vehicle_type": v_type, "color": v_color, "color_votes": {},
                "best_vehicle_crop": v_crop.copy() if v_crop is not None else None,
                "best_plate_crop": None,
                "best_plate_bbox": None,
                "best_vehicle_bbox": [vx1, vy1, vx2, vy2],
                "best_frame_idx": frame_idx,
            }
        else:
            if v_type != "car":
                track_records[tid]["vehicle_type"] = v_type
            if v_color != "other" and track_records[tid]["color"] == "other":
                track_records[tid]["color"] = v_color
            if track_records[tid]["best_text"] is None and v_crop is not None:
                track_records[tid]["best_vehicle_crop"] = v_crop.copy()
                track_records[tid]["best_vehicle_bbox"] = [vx1, vy1, vx2, vy2]

        if v_color != "other":
            track_records[tid]["color_votes"][v_color] = track_records[tid]["color_votes"].get(v_color, 0) + 1

    # only match plates against vehicles big enough to plausibly show a readable plate
    match_candidates = np.array([
        row for row in tracked
        if (row[2] - row[0]) * (row[3] - row[1]) >= MIN_VEHICLE_AREA_FOR_PLATE_PX
    ]) if len(tracked) > 0 else np.empty((0, 5))

    # --- plate detection: ONE call for the whole frame ---
    t0 = time.perf_counter()
    plate_dets = detect_plates_in_frame(frame)
    t_plate_det += (time.perf_counter() - t0)
    plate_detections_run += len(plate_dets)

    # match each detected plate to a vehicle
    t0 = time.perf_counter()
    matched_plate_for_tid = {}
    for pd in plate_dets:
        tid = match_plate_to_vehicle(pd["bbox"], match_candidates) if len(match_candidates) > 0 else None
        if tid is None:
            plate_detections_unmatched += 1
            continue
        if tid not in matched_plate_for_tid or pd["det_conf"] > matched_plate_for_tid[tid]["det_conf"]:
            matched_plate_for_tid[tid] = pd
    t_plate_match += (time.perf_counter() - t0)

    # --- OCR quality gate + read, per matched vehicle ---
    for tid, pd in matched_plate_for_tid.items():
        rec = track_records[tid]
        already_confident = rec["best_conf"] >= OCR_STOP_CONF_THRESHOLD
        maxed_out = rec["n_attempts"] >= MAX_OCR_ATTEMPTS_PER_TRACK
        too_small = pd["area_px"] < MIN_PLATE_AREA_PX

        if already_confident:
            ocr_skipped_already_confident += 1
        elif maxed_out:
            ocr_skipped_max_attempts += 1
        elif too_small:
            ocr_skipped_small_crop += 1
        else:
            px1, py1, px2, py2 = pd["bbox"]
            plate_crop = frame[py1:py2, px1:px2]   # pristine frame
            ocr_calls_run += 1
            t0 = time.perf_counter()
            ocr_result = ocr_plate_crop(plate_crop, pd["det_conf"])
            t_ocr_total += (time.perf_counter() - t0)
            rec["n_attempts"] += 1
            if ocr_result is not None and ocr_result["combined_conf"] > rec["best_conf"]:
                rec["best_text"] = ocr_result["text"]
                rec["best_conf"] = ocr_result["combined_conf"]
                rec["best_timestamp"] = abs_timestamp.isoformat()
                rec["raw_text"] = ocr_result.get("raw_text", ocr_result["text"])
                rec["was_corrected"] = ocr_result.get("was_corrected", False)
                rec["best_frame_idx"] = frame_idx
                rec["best_plate_bbox"] = [int(px1), int(py1), int(px2), int(py2)]
                rec["best_plate_crop"] = plate_crop.copy()
                if tid in tracked_vehicles_in_frame:
                    v_info = tracked_vehicles_in_frame[tid]
                    rec["best_vehicle_bbox"] = v_info["bbox"]
                    if v_info["crop"] is not None:
                        rec["best_vehicle_crop"] = v_info["crop"].copy()
                        if v_info["color"] != "other":
                            rec["color"] = v_info["color"]
                if ocr_result["was_corrected"]:
                    ocr_texts_corrected += 1

    # --- draw overlay on display_frame only ---
    for x1, y1, x2, y2, tid in tracked:
        tid = int(tid)
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(frame_w, x2), min(frame_h, y2)
        if x2 <= x1 or y2 <= y1:
            continue

        rec = track_records[tid]
        cv2.rectangle(display_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)  # green vehicle box

        if tid in matched_plate_for_tid:
            px1, py1, px2, py2 = matched_plate_for_tid[tid]["bbox"]
            cv2.rectangle(display_frame, (px1, py1), (px2, py2), (0, 0, 255), 2)  # red plate box

        label = rec["best_text"] if rec["best_text"] else "..."
        label_full = f"ID{tid} {label}"
        (tw, th), _ = cv2.getTextSize(label_full, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
        cv2.rectangle(display_frame, (x1, y1 - th - 12), (x1 + tw + 6, y1), (255, 255, 255), -1)
        cv2.putText(display_frame, label_full, (x1 + 3, y1 - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)

    t0 = time.perf_counter()
    writer.write(display_frame)
    t_io_write += (time.perf_counter() - t0)

cap.release()
writer.release()
total_loop_end = time.perf_counter()
total_loop_time = total_loop_end - total_loop_start

# Finalize dominant color from votes if available
for tid, rec in track_records.items():
    if rec.get("color_votes"):
        dominant_vote = max(rec["color_votes"], key=rec["color_votes"].get)
        rec["color"] = dominant_vote

# Save crops to disk
crops_saved_count = 0
for tid, rec in track_records.items():
    if rec.get("best_text") is not None:
        safe_plate = re.sub(r'[^A-Za-z0-9]', '_', rec["best_text"])
        if rec.get("best_plate_crop") is not None:
            p_path = os.path.join(PLATE_CROPS_DIR, f"track_{tid:03d}_{safe_plate}.jpg")
            cv2.imwrite(p_path, rec["best_plate_crop"])
            crops_saved_count += 1
        if rec.get("best_vehicle_crop") is not None:
            v_path = os.path.join(VEHICLE_CROPS_DIR, f"track_{tid:03d}_{safe_plate}_vehicle.jpg")
            cv2.imwrite(v_path, rec["best_vehicle_crop"])

print(f"Processed {processed_count} sampled frames (seeked directly, no wasted decode)")
print(f"Tracked {len(track_records)} unique vehicles")
print(f"Saved {crops_saved_count} plate crops to {PLATE_CROPS_DIR}")
print(f"Annotated video written to: {OUTPUT_VIDEO_PATH}")
print()
print("=" * 65)
print(f"[TIMER TELEMETRY] Core video loop wall-clock time: {total_loop_time:.2f}s ({total_loop_time/max(1, processed_count):.4f}s/frame, {processed_count/max(0.001, total_loop_time):.2f} FPS)")
print(f"  - Video read & seek (cv2):       {t_io_read:6.2f}s ({100*t_io_read/max(0.001, total_loop_time):5.1f}%)")
print(f"  - Vehicle detection (YOLOv8):   {t_vehicle_det:6.2f}s ({100*t_vehicle_det/max(0.001, total_loop_time):5.1f}%)")
print(f"  - Tracker updates (SORT):       {t_tracker_update:6.2f}s ({100*t_tracker_update/max(0.001, total_loop_time):5.1f}%)")
print(f"  - Color extraction (HSV):       {t_color_heuristics:6.2f}s ({100*t_color_heuristics/max(0.001, total_loop_time):5.1f}%)")
print(f"  - Plate detection (YOLOv8):     {t_plate_det:6.2f}s ({100*t_plate_det/max(0.001, total_loop_time):5.1f}%)")
print(f"  - Plate-vehicle matching:       {t_plate_match:6.2f}s ({100*t_plate_match/max(0.001, total_loop_time):5.1f}%)")
print(f"  - OCR total (PaddleOCR+CLAHE):  {t_ocr_total:6.2f}s ({100*t_ocr_total/max(0.001, total_loop_time):5.1f}%)")
print(f"  - Video writing (VideoWriter):  {t_io_write:6.2f}s ({100*t_io_write/max(0.001, total_loop_time):5.1f}%)")
print("=" * 65)
print()
print("=== PLATE DETECTION STATS (now one call per FRAME, not per vehicle) ===")
print(f"Plate boxes detected: {plate_detections_run}")
print(f"Plate boxes not matched to any tracked vehicle (dropped): {plate_detections_unmatched}")
print()
print("=== OCR QUALITY GATE STATS ===")
print(f"OCR calls actually run: {ocr_calls_run}")
print(f"OCR skipped — crop too small (< {MIN_PLATE_AREA_PX}px): {ocr_skipped_small_crop}")
print(f"OCR skipped — track already confident (>= {OCR_STOP_CONF_THRESHOLD}): {ocr_skipped_already_confident}")
print(f"OCR skipped — hit max attempts ({MAX_OCR_ATTEMPTS_PER_TRACK}): {ocr_skipped_max_attempts}")
print(f"Reads corrected via char-confusion normalization: {ocr_texts_corrected}")


Processed 300 sampled frames (seeked directly, no wasted decode)
Tracked 138 unique vehicles
Annotated video written to: /content/idahr_v2/output_annotated.mp4

=== PLATE DETECTION STATS (now one call per FRAME, not per vehicle) ===
Plate boxes detected: 502
Plate boxes not matched to any tracked vehicle (dropped): 10

=== OCR QUALITY GATE STATS ===
OCR calls actually run: 261
OCR skipped — crop too small (< 900px): 0
OCR skipped — track already confident (>= 0.9): 0
OCR skipped — hit max attempts (8): 207
Reads corrected via char-confusion normalization: 0


## 11. Export results
Builds `events.json` — one entry per tracked vehicle that got **at least one** successful plate read (tracks with zero readings are skipped rather than fabricated), including the best-confidence text, timestamp, camera metadata, and whether it passed the plate-format regex. Also flags low-confidence reads for manual review, prints a run summary, and plays the annotated video inline.

In [ ]:
# ==============================================================
# CELL 11 — Emit events.json (best-confidence plate per track) +
# zip crops + summary stats
# ==============================================================
import json
import shutil

events = []
event_id = 1
confidences = []
low_conf_flagged = []

for tid, rec in track_records.items():
    if rec["best_text"] is None:
        continue  # never got a single OCR reading for this track — skip, don't fabricate

    cleaned = rec["best_text"]
    regex_valid = bool(re.match(PLATE_REGEX, cleaned))
    confidences.append(rec["best_conf"])

    if rec["best_conf"] < OCR_CONF_FLAG_THRESHOLD:
        low_conf_flagged.append({"track_id": tid, "plate": cleaned, "confidence": rec["best_conf"]})

    events.append({
        "event_id": event_id,
        "camera_id": rec["camera_id"],
        "plate": cleaned,
        "raw_plate": rec.get("raw_text", cleaned),
        "was_corrected": rec.get("was_corrected", False),
        "timestamp": rec["best_timestamp"],
        "confidence": rec["best_conf"],
        "track_id": tid,
        "ocr_attempts": rec["n_attempts"],
        "regex_valid_indian_format": regex_valid,
        "lat": CAMERA_META["lat"],
        "lng": CAMERA_META["lng"],
        "vehicle_type": rec.get("vehicle_type", "car"),
        "color": rec.get("color", "other"),
        "best_frame_idx": rec.get("best_frame_idx"),
        "plate_bbox": rec.get("best_plate_bbox"),
        "vehicle_bbox": rec.get("best_vehicle_bbox"),
    })
    event_id += 1

with open(OUTPUT_EVENTS_PATH, "w") as f:
    json.dump(events, f, indent=2)

# Create zip archives for plate crops and vehicle crops
plate_zip = shutil.make_archive(f"{WORK_DIR}/plate_crops", "zip", PLATE_CROPS_DIR)
vehicle_zip = shutil.make_archive(f"{WORK_DIR}/vehicle_crops", "zip", VEHICLE_CROPS_DIR)
print(f"Zipped plate crops to: {plate_zip}")
print(f"Zipped vehicle crops to: {vehicle_zip}")

print("=== SUMMARY ===")
print(f"Total vehicles tracked: {len(track_records)}")
print(f"Vehicles with at least one plate reading: {len(events)}")
print(f"Average OCR attempts per successful track: "
      f"{sum(e['ocr_attempts'] for e in events)/len(events):.1f}" if events else "N/A")
print(f"Average best-confidence: {sum(confidences)/len(confidences):.3f}" if confidences else "N/A")
print(f"Flagged for manual review (below {OCR_CONF_FLAG_THRESHOLD}): {len(low_conf_flagged)}")
print(f"\nevents.json written to: {OUTPUT_EVENTS_PATH}")

print("\n=== FINAL PLATE READS (spot-check against the video) ===")
for e in events:
    print(f"Track {e['track_id']:3d}: '{e['plate']}' (conf={e['confidence']:.3f}, "
          f"{e['ocr_attempts']} attempts, type={e['vehicle_type']}, color={e['color']})")


=== SUMMARY ===
Total vehicles tracked: 138
Vehicles with at least one plate reading: 35
Average OCR attempts per successful track: 7.1
Average best-confidence: 0.496
Flagged for manual review (below 0.55): 22

events.json written to: /content/idahr_v2/events.json

=== FINAL PLATE READS (spot-check against the video) ===
Track 16: 'FJI4ZHY' (conf=0.61, 8 OCR attempts)
Track 11: 'BGG5USJ' (conf=0.631, 8 OCR attempts)
Track 10: 'NRQ2FKD' (conf=0.531, 8 OCR attempts)
Track 8: 'KHOSZZK' (conf=0.45, 8 OCR attempts)
Track 6: 'GXJ5' (conf=0.468, 8 OCR attempts)
Track 5: 'NSAISAN' (conf=0.563, 8 OCR attempts)
Track 3: 'NAI3NRU' (conf=0.599, 6 OCR attempts)
Track 1: 'APOSJEO' (conf=0.498, 8 OCR attempts)
Track 19: 'LHI3VCY' (conf=0.532, 2 OCR attempts)
Track 25: 'EYGINBG' (conf=0.478, 8 OCR attempts)
Track 23: 'AK64DMV' (conf=0.56, 8 OCR attempts)
Track 33: 'HNI4C' (conf=0.5, 8 OCR attempts)
Track 27: 'OU62HY' (conf=0.299, 8 OCR attempts)
Track 36: 'AYO8HVF' (conf=0.535, 8 OCR attempts)
Track 3

## 12. Download outputs
On Colab this downloads the annotated video and `events.json` directly to your machine. **On Kaggle, use the notebook's Output/Data tab after running instead** — the cell below is wrapped in a `try/except` so it won't error out the run if `google.colab` isn't available.

In [ ]:
# ==============================================================
# CELL 12 — Download the annotated video + events.json (Colab only)
# ==============================================================
try:
    from google.colab import files
    files.download(OUTPUT_VIDEO_PATH)
    files.download(OUTPUT_EVENTS_PATH)
except (ImportError, NotImplementedError, Exception):
    print("Not running on Colab — skipping auto-download. "
          f"Grab {OUTPUT_VIDEO_PATH} and {OUTPUT_EVENTS_PATH} from the "
          "Kaggle notebook's Output tab after the session finishes.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>